# Olist E-Commerce Data Profiling

## Objective

This notebook evaluates the raw Olist e-commerce datasets before any cleaning or transformation is performed.

The profiling process will:

- Inspect dataset dimensions, columns, and data types
- Identify missing and duplicate values
- Evaluate candidate primary and foreign keys
- Examine relationships between datasets
- Test referential integrity
- Identify potential data quality issues
- Examine business-relevant categorical and numerical fields

The findings from this analysis will guide the subsequent data cleaning, transformation, and dimensional modelling stages of the project.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

## Loading Data

In [2]:
RAW_DATA_DIR = Path("../data/raw")
# glob() looks inside RAW_DATA_DIR and finds anything matching
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

"""
for file in csv_files:
    print(file.name)
"""

dataset_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

datasets = {}

# Reading with pandas, storing in datasets dictionary with name as key
for name, filename in dataset_files.items():
    file_path = RAW_DATA_DIR / filename
    dataframe = pd.read_csv(file_path)
    datasets[name] = dataframe

# datasets.keys()

## Inspecting Data

In [3]:
summary_data = []

for name, df in datasets.items():
    summary_data.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
    })

dataset_summary = pd.DataFrame(summary_data)
dataset_summary = dataset_summary.sort_values("rows", ascending=False)

print(dataset_summary)

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print(f"Shape: {df.shape}")
    print("Columns:")
    
    for column in df:
        print(f"  - {column}")

                dataset     rows  columns
1           geolocation  1000163        5
2           order_items   112650        7
3              payments   103886        5
0             customers    99441        5
5                orders    99441        8
4               reviews    99224        7
6              products    32951        9
7               sellers     3095        4
8  category_translation       71        2

customers
--------------------------------------------------
Shape: (99441, 5)
Columns:
  - customer_id
  - customer_unique_id
  - customer_zip_code_prefix
  - customer_city
  - customer_state

geolocation
--------------------------------------------------
Shape: (1000163, 5)
Columns:
  - geolocation_zip_code_prefix
  - geolocation_lat
  - geolocation_lng
  - geolocation_city
  - geolocation_state

order_items
--------------------------------------------------
Shape: (112650, 7)
Columns:
  - order_id
  - order_item_id
  - product_id
  - seller_id
  - shipping_limit_date
  

In [4]:
datasets["orders"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
datasets["customers"].head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [6]:
for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print("-" * 50)
    print(df.dtypes)


CUSTOMERS
--------------------------------------------------
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

GEOLOCATION
--------------------------------------------------
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

ORDER_ITEMS
--------------------------------------------------
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

PAYMENTS
--------------------------------------------------
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int

### Dataset Inventory Findings

The raw data consists of nine related datasets ranging from 71 records in the category translation table to over 1 million records in the geolocation dataset.

Initial observations:

- The `orders` and `customers` datasets both contain 99,441 rows. Further key analysis is required to determine the relationship between orders, customer records, and unique customers.
- The `order_items` dataset contains 112,650 rows compared with 99,441 orders, suggesting that some orders contain multiple line items. This will be verified through key analysis.
- The `payments` dataset contains 103,886 rows, exceeding the number of orders. This suggests that some orders may be associated with multiple payment records.
- The `reviews` dataset contains 99,224 rows, slightly fewer than the number of orders. Review coverage and the relationship between reviews and orders will require further investigation.
- The `geolocation` dataset contains over 1 million records, substantially more than the other datasets. Its grain and the uniqueness of ZIP code prefixes must be investigated before using it in joins.

These observations are preliminary and will be validated through duplicate, key, and relationship analysis.

### Schema and Data Type Findings

Initial schema inspection shows that the datasets contain a combination of string, integer, and floating-point fields representing identifiers, geographic information, product attributes, monetary values, and timestamps.

Key observations:

- Identifier fields such as `customer_id`, `order_id`, `product_id`, and `seller_id` are stored as strings, which is appropriate because they function as identifiers rather than numerical measures.
- Geographic ZIP code prefixes are currently represented as integers in the customer, seller, and geolocation datasets. Their treatment will be evaluated during the transformation stage because ZIP codes represent categorical location identifiers rather than quantities.
- Monetary fields including `price`, `freight_value`, and `payment_value` are represented as floating-point values.
- Several product attributes, including weight and physical dimensions, are represented as floating-point values.
- All date and timestamp fields are currently represented as strings. These include `order_purchase_timestamp`, `order_approved_at`, delivery timestamps, `shipping_limit_date`, and review timestamps. These fields will require explicit datetime conversion during data transformation.
- Review scores and payment installment counts are represented as integers.
- Some column names in the products dataset use the spelling `lenght` rather than `length`. These names originate from the source dataset and will be considered for standardization during the cleaning stage.

No data types have been modified at this stage. The purpose of this assessment is to document the source schema before transformation.

## Check for missing data

In [7]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isnull().sum()

        if missing_count > 0:
            missing_summary.append({
                "dataset": name,
                "column": column,
                "missing_count": missing_count,
                "missing_percent": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_summary = pd.DataFrame(missing_summary)

missing_summary.sort_values("dataset",ascending=False)

,dataset,column,missing_count,missing_percent
0,reviews,review_comment_title,87656,88.34
1,reviews,review_comment_message,58247,58.70
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01
10,products,product_length_cm,2,0.01
11,products,product_height_cm,2,0.01
12,products,product_width_cm,2,0.01


### Missing Value Findings

Missing values are concentrated in a small number of fields rather than being distributed uniformly throughout the source data.

#### Reviews

- `review_comment_title` is missing for 88.34% of review records.
- `review_comment_message` is missing for 58.70% of review records.
- These fields appear to be optional textual components of a review rather than required identifiers or measures. Their absence will therefore not automatically be treated as a data-quality failure.

#### Orders

- `order_delivered_customer_date` is missing for 2.98% of orders.
- `order_delivered_carrier_date` is missing for 1.79% of orders.
- `order_approved_at` is missing for 0.16% of orders.
- These missing timestamps may be related to the lifecycle or status of an order. The relationship between `order_status` and missing order timestamps will be investigated before determining how these records should be handled.

#### Products

- `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` each contain 610 missing values (1.85% of product records).
- The identical missing-value counts suggest that these attributes may be missing together for the same products. This will be tested during further profiling.
- Product weight, length, height, and width each contain only 2 missing values (0.01%), representing a very small proportion of the product dataset.

At this stage, no missing values have been removed, filled, or otherwise modified. Further analysis will determine whether missing values represent optional information, expected business states, or data-quality issues requiring transformation.

## Check if the data missing in `products` is the same

In [8]:
product_missing_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

# isnull(): Panda will change every value in the 4 columns to true or false
# .all(axis=1): Check if the entry is true for all 4 columns per row
# Sum up the rows that are true (null) across the missing columns
print(datasets["products"][product_missing_columns].isnull().all(axis=1).sum())

# value_counts() checks how many items had 0-4 columns missing. With this way you can check if some rows had only
# some attrtibutes missing
print(datasets["products"][product_missing_columns].isnull().value_counts())

product_missing_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print(datasets["products"][product_missing_columns].isnull().value_counts())

610
product_category_name  product_name_lenght  product_description_lenght  product_photos_qty
False                  False                False                       False                 32341
True                   True                 True                        True                    610
Name: count, dtype: int64
product_weight_g  product_length_cm  product_height_cm  product_width_cm
False             False              False              False               32949
True              True               True               True                    2
Name: count, dtype: int64


Missing-value analysis identified two consistent patterns in the `products` dataset:

- Returns 610 therefore all 610 rows in products have `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` as null

- Returns 2 therefore both rows are missing `product_weight_g`, `product_length_cm`, `product_height_cm`, and `product_width_cm`

These patterns suggest that the missing product attributes are grouped by record rather than occurring independently across different products. The affected records will be investigated further before determining how they should be handled during data cleaning.

## Investigating the missing `orders` timestamps

In [ ]:
print(datasets["orders"]["order_status"].value_counts())


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
order_status
delivered      97.02
shipped         1.11
canceled        0.63
unavailable     0.61
invoiced        0.32
processing      0.30
created         0.01
approved        0.00
Name: proportion, dtype: float64


- `order_delivered_customer_date` is missing for 2.98% of orders. 2965 in total
- `order_delivered_carrier_date` is missing for 1.79% of orders. 1783 in total
- `order_approved_at` is missing for 0.16% of orders. 160 in total

96478 orders got delivered. Let's check how many of those delivered orders have `order_delivered_customer_date`, `order_delivered_carrier_date`. or `order_approved_at` missing.

In [19]:
delivered_orders = datasets["orders"][datasets["orders"]["order_status"] == "delivered"]
print(len(delivered_orders))

print(delivered_orders[["order_delivered_carrier_date","order_delivered_customer_date","order_approved_at"]].isnull().sum())
print(delivered_orders[["order_delivered_carrier_date","order_delivered_customer_date","order_approved_at"]].isnull().value_counts().reset_index(name="count"))

96478
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_approved_at                14
dtype: int64
   order_delivered_carrier_date  order_delivered_customer_date  \
0                         False                          False   
1                         False                          False   
2                         False                           True   
3                          True                          False   
4                          True                           True   

   order_approved_at  count  
0              False  96455  
1               True     14  
2              False      7  
3              False      1  
4              False      1  


- 96,455 delivered orders had all three analyzed timestamps present.
- 14 `delivered` orders had only `order_approved_at` missing.
- 7 `delivered` orders had only 7 `order_delivered_customer_date` missing.
- 1 `delivered` order had both `order_delivered_customer_date` and `order_delivered_carrier_date` missing but had `order_approved_at`.
- 1 `delivered` order had only `order_delivered_carrier_date` missing.
- In total, 23 delivered orders were missing at least one expected lifecycle timestamp.

Because these orders are marked as `delivered`, the missing timestamps may represent incomplete event tracking or missing historical data. The source data does not provide enough information to determine the exact cause.

These records will not be removed during profiling. Their suitability will instead be evaluated based on the downstream analysis. For example, an order with a missing customer delivery date may still be useful for sales analysis but cannot be used to calculate customer delivery time.

## Checking uniqueness of `order_id`, `customer_id`, and `product_id`

In [32]:
print("Orders:", len(datasets["orders"]))
print("Unique order IDs:", datasets["orders"]["order_id"].nunique())
print("Duplicate order IDs:", datasets["orders"]["order_id"].duplicated().sum())
print("Unique customer IDs:",datasets["orders"]["customer_id"].nunique())
print()

print("Products:", len(datasets["products"]))
print("Unique product IDs:", datasets["products"]["product_id"].nunique())
print("Duplicate product IDs:", datasets["products"]["product_id"].duplicated().sum())

Orders: 99441
Unique order IDs: 99441
Duplicate order IDs: 0
Unique customer IDs: 99441

Products: 32951
Unique product IDs: 32951
Duplicate product IDs: 0


### Grain is the exact level of detail that a single row represents in a table

### Orders
- Grain: one row per order
- Candidate primary key: order_id
- Another finding that's important in the `orders` dataset is that the `customer_id` is unique. We can investigate this with the `customers` dataset and check the relation between `customer_id` and `customer_unique_id` in that dataset. This can give valuable insight into our loyal customers.

### Products
- Grain: one row per product
- Candidate primary key: product_id


## Check timestamps chronology

In [ ]:
""" .apply() applies a function to each selected column
pd.to_datetime converts text containing datetimes into recognized dates and time by pandas.
This will allow us to extract parts of dates using .year, .month, .day, .hour, .day_name() i.e
check which day of the week customers place the most orders.

Find date ranges using .min(), .max(). i.e show oldest sold product, whats trending

Subtract 2 different datetimes. i.e check delivery times, late orders
"""
order_dates = datasets["orders"][
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].apply(pd.to_datetime)

order_dates = datasets["orders"][
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
].apply(pd.to_datetime)

print(
    "Orders approved before purchase:",
    (
        order_dates["order_approved_at"]
        < order_dates["order_purchase_timestamp"]
    ).sum()
)

print(
    "Orders sent to carrier before approval:",
    (
        order_dates["order_delivered_carrier_date"]
        < order_dates["order_approved_at"]
    ).sum()
)

print(
    "Orders delivered to customer before carrier date:",
    (
        order_dates["order_delivered_customer_date"]
        < order_dates["order_delivered_carrier_date"]
    ).sum()
)

print(
    "Orders delivered after estimated delivery date:",
    (
        order_dates["order_delivered_customer_date"]
        > order_dates["order_estimated_delivery_date"]
    ).sum()
)

print(
    "Earliest order purchase:",
    order_dates["order_purchase_timestamp"].min()
)

print(
    "Latest order purchase:",
    order_dates["order_purchase_timestamp"].max()
)

Orders approved before purchase: 0
Orders sent to carrier before approval: 1359
Orders delivered to customer before carrier date: 23
Orders delivered after estimated delivery date: 7827
Earliest order purchase: 2016-09-04 21:15:19
Latest order purchase: 2018-10-17 17:30:18


Order lifecycle timestamps were converted to Pandas datetime values temporarily for profiling so chronological relationships could be evaluated.

- 0 orders were approved before the recorded purchase timestamp.
- 1,359 orders have an `order_delivered_carrier_date` earlier than `order_approved_at`.
- 23 orders have an `order_delivered_customer_date` earlier than `order_delivered_carrier_date`.
- 7,827 orders were delivered after their estimated delivery date.
- The earliest recorded purchase occurred on September 4, 2016.
- The latest recorded purchase occurred on October 17, 2018.

The 1,359 carrier-before-approval records and 23 customer-before-carrier records represent potential chronological inconsistencies in the recorded order lifecycle. These records will be investigated further before determining whether they reflect unusual business processes, timestamp recording issues, or data-quality problems.

The presence of 7,827 orders delivered after the estimated delivery date also indicates that the dataset can support fulfillment metrics such as late-delivery rate, delivery duration, and average days early or late.